<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Textual_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4 Lab: Transformers for Text Analysis (original case)

> **Original case of the book, corrected in 2026.** This is the case study printed in Chapter 4, which predicts next month's Chicago Fed National Activity Index from Federal Reserve speeches. The current Chapter 4 lab is a new case on the monetary policy stance of the same speeches: [open it in Colab](https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Fed_Stance.ipynb) or [view it on GitHub](https://github.com/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Fed_Stance.ipynb). This case still shows how to train a transformer on text, step by step. When the speeches are split by time instead of at random, however, its model does no better than chance, as the usability test at the end of this notebook shows. The changes made to this notebook are listed in the Errata at the end.

In this lab, we will explore Transformer models for text analysis using the [Hugging Face Transformers library](https://huggingface.co/docs/transformers/index). This library provides a wide range of pre-trained models that we can leverage for various natural language processing (NLP) tasks.


First, we need to install and import the necessary libraries.

In [ ]:
# Install necessary packages, if not done before
!pip install transformers==5.18.0 evaluate accelerate

## **Downloading Datasets**

Now, we will use two datasets for this lab:

1. **Federal Reserve Speeches (1996–2024)**  
   - This dataset contains the **text** of speeches given mostly by Federal Reserve officials over the years. A small share (roughly 4%) are by officials of other central banks, such as the European Central Bank or the Bank of England, many of them given at Fed-hosted events such as the Jackson Hole symposium.  

2. **[Chicago Fed National Activity Index (CFNAI)](https://fred.stlouisfed.org/series/CFNAI)**  
   - The CFNAI is a comprehensive economic indicator that tracks **85 key economic factors** such as employment, production, and consumption.  
   - It helps measure national economic activity:
     - **Zero value** → Economy is growing at its historical trend rate.
     - **Negative values** → Below-average growth.
     - **Positive values** → Above-average growth.

In [ ]:
# Bare file ids work with every gdown version (--fuzzy was removed in gdown 6).
!gdown -q 1uVt9BC2tgr-MWrFZvYvA_I8IzTabNZtL    # fed_speeches.csv
!gdown -q 1I7isSks6Y8kJoigbDZJumgdQ1fFpwn4i    # CFNAI.csv

In [ ]:
# Imports
import numpy as np
import pandas as pd
import random
# Scikit-learn
from sklearn.metrics import roc_auc_score, roc_curve, auc

# Plots
import matplotlib.pyplot as plt
%matplotlib inline

# Import PyTorch libraries
import torch
import torch.nn as nn

# Huggingface
import transformers
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import set_seed
from datasets import Dataset, DatasetDict
import evaluate

We **set a fixed random seed** at the start, before any data is split or any model is created, so that the results are reproducible.

In [ ]:
# Set a fixed seed value
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

Now, we will load the Federal Reserve Speeches dataset.
Once loaded, we will inspect the structure of the dataset. It has two columns: `date`, the date of each speech, and `text`, the full text of the speech. This will help us determine how to preprocess and analyze the text data effectively.

In [ ]:
fed_speech = pd.read_csv("/content/fed_speeches.csv", delimiter=",", on_bad_lines="skip", engine="python")

In [ ]:
fed_speech.info()

In [ ]:
fed_speech.head(10)

## Data preprocess - merging

In this step, we will preprocess the Federal Reserve Speeches dataset to prepare it for analysis. First, we will extract the **year** and **month** from the speech dates. Then, we will shift the month forward by one to align each speech with the economic conditions of the following month.

Our goal is to predict whether next month's economic growth will be above or below average, based on the language used in Federal Reserve speeches. This adjustment ensures that our model learns from past speeches to forecast future economic trends more effectively.

In [ ]:
# Create year and month columns
fed_speech['date'] = pd.to_datetime(fed_speech['date'], errors='coerce')
fed_speech["year"] = fed_speech["date"].dt.year  # Extract year
fed_speech["month"] = fed_speech["date"].dt.month  # Extract month

# Shift the month forward by 1
fed_speech["month"] += 1

# Handle December (12 → 1 and increase year)
fed_speech.loc[fed_speech["month"] == 13, "month"] = 1
fed_speech.loc[fed_speech["month"] == 1, "year"] += 1

A few speeches in the dataset have the **wrong date**: the date printed at the top of the speech differs from the `date` column, in most cases by exactly one year. A wrong date matches the speech to the wrong month of CFNAI and can flip its label. For example, a speech from February 2020 that is dated February 2021 would be labelled with the CFNAI for March 2021 (strong growth) instead of March 2020 (the COVID-19 crash). We drop these seven speeches.

In [ ]:
# Speeches whose date in the file is wrong, with the date the speech was actually given
wrong_dates = [
    "2002-12-18",  # Meyer, given 2001-12-18
    "2004-11-03",  # Greenspan, given 2003-10-29
    "2004-11-12",  # Greenspan, given 2003-11-06
    "2004-11-20",  # Ferguson, given 2003-11-13
    "2004-11-23",  # Greenspan, given 2003-11-20
    "2021-02-21",  # Clarida, given 2020-02-21
    "2024-12-08",  # Bowman, given 2024-01-08
]

# Compare calendar days only (some dates in the file also include a time)
wrong_rows = fed_speech["date"].dt.normalize().isin(pd.to_datetime(wrong_dates))
print(f"Dropping {wrong_rows.sum()} speeches with wrong dates")
fed_speech = fed_speech[~wrong_rows]

Next, we will load the Chicago Fed National Activity Index (CFNAI) dataset, which serves as a key indicator of U.S. economic activity.

Once loaded, we will merge the CFNAI dataset with the Federal Reserve Speeches dataset using year and month as the merging keys. Since we previously adjusted the speech dataset by shifting the month forward, this ensures that each speech is aligned with the economic activity of the following month.

In [ ]:
econ_index = pd.read_csv('/content/CFNAI.csv')
econ_index.columns = ['date', 'CFNAI']
econ_index['date'] = pd.to_datetime(econ_index['date'], errors='coerce')
econ_index["year"] = econ_index["date"].dt.year  # Extract year
econ_index["month"] = econ_index["date"].dt.month  # Extract month
econ_index.drop(columns=['date'], inplace=True)
econ_index

In [ ]:
merged_df = fed_speech.merge(econ_index, on=["year", "month"], how="left")
merged_df

## **Text Preprocessing: Tokenization, Stopword Removal, and Cleaning**

In this step, we preprocess the speech text by **tokenizing, removing stopwords, and eliminating punctuation** to prepare the data for further analysis.

### **1. Import Required Libraries**
The following libraries from **NLTK (Natural Language Toolkit)** are used:
- `word_tokenize` → Splits text into individual words (tokens).
- `stopwords` → Provides a list of common English stopwords (e.g., "the", "is", "and").

Punctuation is removed by keeping only tokens made of letters and digits (`isalnum()`).

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re


nltk.download('stopwords')
nltk.download('punkt_tab')
stop_words = set(stopwords.words("english"))


def clean_text(text):
    text = text.lower()  # Convert to lowercase
    tokens = word_tokenize(text)  # Tokenize text
    filtered_tokens = [
        token for token in tokens
        if token.isalnum() and token not in stop_words  # Keep only letters/digits (drops punctuation) and remove stop words
    ]
    cleaned_text = ' '.join(filtered_tokens)
    return cleaned_text

merged_df['text_cleaned'] = merged_df['text'].apply(clean_text)

We also **remove dates** from the text, such as years ("2024") and dates like "January 8" or "3 December". A date tells the model *when* a speech was given, not *what* it says, and the model could learn to recognize the period instead of the content. Month names that are not part of a date are kept (for example, "may" is usually the verb).

In [ ]:
MONTHS = "january|february|march|april|may|june|july|august|september|october|november|december"

def remove_dates(text):
    text = re.sub(rf"\b({MONTHS}) \d{{1,2}}\b", " ", text)    # "january 8"
    text = re.sub(rf"\b\d{{1,2}} ({MONTHS})\b", " ", text)    # "3 december"
    text = re.sub(rf"\b({MONTHS}) (19|20)\d\d\b", " ", text)  # "december 2024"
    text = re.sub(r"\b(19|20)\d\ds?\b", " ", text)            # "2024", "1990s"
    return " ".join(text.split())

merged_df['text_cleaned'] = merged_df['text_cleaned'].apply(remove_dates)

As we discuss in the book, you would need to test the best performance depending on the cleaning steps necessary. Normally, either doing nothing, removing stopwords, or lowercase would lead to the best performance depending on the application area. Combining strategies may lead to decreased performance. Test different strategies and see if you can improve this model!

You can inspect how the **`clean_text`** function processed the text by viewing the cleaned version stored in the **`text_cleaned`** column.

In [ ]:
merged_df.tail(5)

As observed above, rows corresponding to **January 2025 (2025-01)** do not have CFNAI values. To address this, we will extract these rows and set them aside. At the end of the lab, we will use them to predict January 2025.


In [ ]:
# Extract the last two rows that contain NaN values
nan_rows_df = merged_df[merged_df.isna().any(axis=1)]

# Remove these rows from the main DataFrame
merged_df = merged_df.drop(nan_rows_df.index)

## **Visualizing Frequent Words in FED Speeches with a Word Cloud**  

In this step, we generate a **word cloud** to visualize the most frequently used words in the **Federal Reserve speeches dataset**. A word cloud is a useful tool for quickly identifying common terms in text data.


In [ ]:
from wordcloud import WordCloud

sample_txt = " ".join(i for i in merged_df['text_cleaned'])

wc = WordCloud(colormap="Set2", collocations=False, random_state=SEED).generate(sample_txt)
plt.title("Most Frequent Words in FED Speeches")
plt.axis("off")
plt.imshow(wc,interpolation='bilinear')
plt.show()

Pretty cool! We can see that the Federal Reserve frequently uses words like financial, bank, market, rate and policy, as well as risk and inflation.

## Labelling

In this step, we categorize the **Chicago Fed National Activity Index (CFNAI)** values into binary labels to prepare our dataset for classification.

---

### **1. Define Bins and Labels**  
We create two categories based on the CFNAI values:  
- **Negative or zero CFNAI (`≤ 0`) → Label 1**  
- **Positive CFNAI (`> 0`) → Label 0**  


In [ ]:
# Define bins and labels
bins = [-float('inf'), 0, float('inf')]
labels = [1, 0]

# Apply categorization
merged_df["label"] = pd.cut(
    merged_df["CFNAI"], bins=bins, labels=labels, include_lowest=True
)

# Convert to integer type
merged_df["label"] = merged_df["label"].astype(int)

merged_df

The counts below show that the labels are fairly balanced: 1,249 speeches have label 1 and 1,205 have label 0.

In [ ]:
merged_df.label.value_counts()

Now that we have cleaned and processed the dataset, we will save it for future use.


In [ ]:
# Save to CSV
merged_df.to_csv('FEDSpeechesProcessed.csv', index=False)

## **Preparing the Dataset for Model Training**  

In this step, we **convert the preprocessed dataset into a Hugging Face `Dataset` format**, encode the labels, and split the data into training, validation and test sets.

---

We extract the **cleaned text (`text_cleaned`)** and its corresponding **label (`label`)** from `merged_df`, then convert it into a Hugging Face `Dataset`.

We split the speeches **at random** into three sets, keeping the same share of each label in every set:

- **Training (70%):** used to train the model.
- **Validation (15%):** after every epoch, we measure the model's AUC on this set and keep the epoch with the highest AUC.
- **Test (15%):** used only at the end, for the ROC curve and AUC.

At the end of the lab, we also use the model on the December 2024 speeches that we set aside earlier, to predict January 2025.

Note that all speeches from the same month share the same label, so a random split puts speeches from the same month in different sets. This can make the test results look better than the model would do on future speeches; splitting the data by date avoids this.

In [ ]:
# Create the dataset
# preserve_index=False: don't add the pandas row index to the dataset as an extra column.
fed_speech_data = Dataset.from_pandas(merged_df.loc[:,['text_cleaned', 'label']], preserve_index=False)

# Set the label variable
fed_speech_data = fed_speech_data.class_encode_column("label")

# Random split: 70% train, 15% validation, 15% test, with the same share of each label in every set
train_rest = fed_speech_data.train_test_split(test_size=0.3, stratify_by_column="label", seed=SEED)
validation_test = train_rest["test"].train_test_split(test_size=0.5, stratify_by_column="label", seed=SEED)
fed_speech_data = DatasetDict({
    "train": train_rest["train"],
    "validation": validation_test["train"],
    "test": validation_test["test"],
})
fed_speech_data

In [ ]:
fed_speech_data['train'].features

## **Tokenizing the Text Data**  

Before feeding our text data into a Transformer model, we need to **tokenize** it. Tokenization converts raw text into numerical format that the model can understand.

---

We use the **DistilBERT tokenizer** from Hugging Face’s `transformers` library.

DistilBERT can read at most **512 tokens** (word pieces) at a time, so `truncation=True` keeps only the first 512 tokens of each speech. Almost all speeches (96%) are longer than that: a typical speech has about 1,900 tokens, so the model sees only roughly its first quarter (the title, the speaker and venue, and the opening paragraphs). This limits how much of each speech the model can use.


In [ ]:
# Tokenize the data.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
# Tokenize each speech; truncation=True keeps only the first 512 tokens (DistilBERT's limit).
def preprocess_function(examples):
    return tokenizer(examples["text_cleaned"], truncation=True)

In [ ]:
tokenized_fed_speech_data = fed_speech_data.map(preprocess_function, batched=True)

In [ ]:
# Save the outcome to disk to not run this again.
tokenized_fed_speech_data.save_to_disk("TokenizedData")

## **Applying Data Collation for Efficient Batching**  

When working with Transformer models, input sequences need to be **padded** to the same length within a batch. To handle this efficiently, we use a **data collator**.

---


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# How many classes there are.
num_labels = len(merged_df["label"].unique())
print(f'There are {num_labels} classes in the dataset.')

Now that we have tokenized the text data, we need to define the **Transformer model** that will be used for classification.



In [ ]:
# Reset the seed so the new classification layer gets the same starting weights,
# even if this cell is run again.
set_seed(SEED)

# attn_implementation="eager": a simpler attention calculation that gives repeatable
# results with full_determinism=True (see the training arguments).
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels, attn_implementation="eager"
    )

We define the **evaluation metrics** to assess the performance of our model on the validation set: **accuracy**, the proportion of correctly classified samples, and **AUC**, which measures how well the model ranks class 1 speeches above the others (0.5 is a coin toss, 1.0 is perfect). We use AUC to pick the best epoch.


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    # Probability of class 1, which the AUC needs
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "auc": roc_auc_score(labels, probs),
    }

## **Defining Training Parameters**  

Now, we configure the **training arguments** that determine how our model will be trained using the Hugging Face `Trainer` API.

---

In [ ]:
training_args = TrainingArguments(
    # Where to store the model.
  output_dir="ModelOutput",
    # Learning rate to use.
    learning_rate=1e-5,
    # Batch size to use per GPU in training.
    per_device_train_batch_size=32,  ## T4 (free Colab GPU)
    #per_device_train_batch_size=200,  ## A100 (high-RAM); runs out of memory on a T4
    # Batch size to use per GPU in evaluation
    per_device_eval_batch_size=32,   ## T4 (free Colab GPU)
    #per_device_eval_batch_size=200,  ## A100 (high-RAM)
    # Epochs to train
    num_train_epochs=15,
    # If decaying or not the weights
    weight_decay=5e-4,
    # When to evaluate the model
    eval_strategy="epoch",
    # When to save checkpoint
    save_strategy="epoch",
    # After training, load the epoch with the highest validation AUC (not the last epoch).
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
    # Save in Huggingface? (Account required)
    push_to_hub=False,
    # Log the training loss once per epoch
    logging_strategy="epoch",
    # Random seed used by the Trainer (set at the start of the notebook)
    seed=SEED,
    # Make GPU calculations repeatable (slower, but the same result every run)
    full_determinism=True,
    # Experiment tracking: "none" turns it off; use "wandb" for Weights & Biases (account required).
    report_to="none",
)

We initialize the Hugging Face `Trainer` for model training. The random seed was fixed at the start of the notebook, and `full_determinism=True` makes the GPU calculations repeatable, so running the training again on the same type of GPU gives the same result.

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Empty VRAM
torch.cuda.empty_cache()

# Create trainer object.
trainer = Trainer(
    # What model to use.
    model=model,
    # Arguments to the model
    args=training_args,
    # Training data
    train_dataset=tokenized_fed_speech_data["train"],
    # Validation data, used to pick the best epoch
    eval_dataset=tokenized_fed_speech_data["validation"],
    # How to pad sequences
    data_collator=data_collator,
    # Error function
    compute_metrics=compute_metrics,
)

Now that we have set up the **dataset, tokenizer, model, training arguments, and `Trainer`**, it's time to **train the model**!  

After every epoch, the `Trainer` shows the training loss and the validation loss, accuracy and AUC in a table. If you also want to track experiments online with **[Weights & Biases (W&B)](https://wandb.ai/home)**, set `report_to="wandb"` in the training arguments and sign in with your W&B API key.

During training you may see a `UserWarning` about the CUBLAS workspace size. It comes from `full_determinism=True` and can be safely ignored.



In [ ]:
trainer.train()

The table above shows how training went. The training loss falls from 0.69 in the first epoch to 0.35 in the last. The validation AUC rises from 0.63 after the first epoch to its highest value, **0.71 at epoch 12**, and then levels off. Meanwhile, the validation loss is lowest at epoch 5 (0.63) and rises to 0.72 by epoch 15, a sign that the model is starting to overfit the training speeches. Because we set `load_best_model_at_end=True`, the `Trainer` keeps the model from epoch 12 rather than the last epoch.

The numbers in this notebook come from a run on a free Colab T4 GPU. On another type of GPU (for example, an A100) the same code gives somewhat different numbers, even with `full_determinism=True`.


After training, we need to **save the model** so we can reuse it for evaluation, inference, or further fine-tuning without retraining from scratch.



In [ ]:
# Save the model to a folder
trainer.save_model('FEDSpeechModel')

In [ ]:
# Zip it
!zip -r DistilBert.zip FEDSpeechModel

In [ ]:
# Calculate AUC over the test set.
predictions = trainer.predict(tokenized_fed_speech_data["test"])
preds = predictions.predictions

# The model outputs raw scores (logits) for both classes.
# Convert them to the probability of class 1, which is what the ROC curve needs.
scores = torch.softmax(torch.tensor(preds), dim=1)[:, 1].numpy()

# Plot ROC Curve
fpr, tpr, threshold = roc_curve(tokenized_fed_speech_data["test"]["label"], scores)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.savefig('c4_ROC_Curve.pdf')
plt.show()


On the test set, which the model never saw during training or when we picked the best epoch, the AUC is **0.61**. This is better than a coin toss (0.5) but far from perfect (1.0), and lower than the best validation AUC (0.71). Some of that gap is expected: we picked the epoch that did best on the validation set, so the validation AUC is slightly optimistic. Keep in mind also that, with a random split, speeches from the same month can appear in both the training and test sets, so even 0.61 may overstate how well the model would predict future months.

## **Using the Model on the Latest FED Speeches**  

Now that we have trained and tested our model, let's use it on the **latest Federal Reserve speeches**, from December 2024, to predict whether January 2025 will see below-average growth.

---


In [ ]:
text = nan_rows_df.iloc[0]['text_cleaned']
text

In [ ]:
# Apply tokenizer and return pytorch tensors
inputs = tokenizer(text, return_tensors="pt", truncation=True)

In [ ]:
model.eval()  # switch off dropout for prediction
with torch.no_grad():
    outputs = model(**inputs.to(model.device))
    logits = outputs.logits

In [ ]:
# Probabilities
probs = nn.functional.softmax(logits, dim=1).cpu().numpy()
print(probs)

# Class
print(f'The text is predicted to be of class {np.argmax(probs)}')

Above is the model's prediction for January 2025, based on a Federal Reserve speech from December 2024: Adriana Kugler's speech of December 3, 2024. The model predicts **class 1** (below-average growth) with a probability of 0.86. A confident prediction is not necessarily a correct one: remember that the test AUC is only 0.61.

How did the model do? The CFNAI for January 2025 is [-0.50](https://fred.stlouisfed.org/series/CFNAI) (first reported as -0.03 and later revised), which is class 1 in our analysis, so the prediction is correct. The dataset has one other December 2024 speech, by Christopher Waller; you can score it by changing `iloc[0]` to `iloc[1]`.


In [ ]:
# Optional: keep the trained model. Colab deletes /content when the session ends, so copy the
# zip to your own Google Drive. Remove the # signs below to do it.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp DistilBert.zip /content/drive/MyDrive/

## **Usability Test: A Split by Time**

The notebook so far shows how to load text, preprocess it and fine-tune a transformer, and those steps apply to any text task. Whether the trained model is usable is a different question. Its test AUC looks useful, and the book reports a higher one, but before a bank relies on such a model two questions matter. Is the text better than a number the bank already has? And does the model still work on speeches given after the ones it learned from?

We answer both with three checks, starting with what the random split puts in the test set. We then compare the model with the Chicago Fed National Activity Index of two months before each speech, a number that is public when the speech is given. Finally, we train the same model again, this time on the speeches of 1996 to 2016, choose its best epoch on 2017 to 2019, and test it on 2020 to 2024. The last check trains DistilBERT once more, so it takes about as long as the training above.

In [ ]:
# Rebuild the random split of Section "Preparing the Dataset" with a row number, to see which speeches fell where.
with_rows = Dataset.from_pandas(merged_df[['text_cleaned', 'label']].assign(row=np.arange(len(merged_df))),
                                preserve_index=False).class_encode_column('label')
check_rest = with_rows.train_test_split(test_size=0.3, stratify_by_column='label', seed=SEED)
check_vt = check_rest['test'].train_test_split(test_size=0.5, stratify_by_column='label', seed=SEED)
assert check_vt['test']['text_cleaned'] == fed_speech_data['test']['text_cleaned'], 'not the same split as above'

random_train = merged_df.iloc[check_rest['train']['row']]
random_test = merged_df.iloc[check_vt['test']['row']]
train_months = set(zip(random_train['year'], random_train['month']))
shared = np.mean([(y, m) in train_months for y, m in zip(random_test['year'], random_test['month'])])
print(f'{shared:.0%} of the test speeches have a target month that also appears in the training set.')

Under the random split, almost every test speech has a "twin" in the training set: another speech whose label is the CFNAI of the same month, and therefore the same label. A model that recognizes *when* a speech was given, from its topics, names or events, can then predict its label without understanding what the speech says about the economy.

In [ ]:
# The CFNAI of two months before each speech: public on the day of the speech.
cfnai = econ_index.assign(period=pd.to_datetime(econ_index[['year', 'month']].assign(day=1)).dt.to_period('M')).set_index('period')['CFNAI']
merged_df['cfnai_lag2'] = merged_df['date'].dt.to_period('M').map(lambda p: cfnai.get(p - 2, np.nan))
random_test = merged_df.loc[random_test.index]

# A low index means a weak economy, so minus the index ranks speeches by the chance of class 1.
ok = random_test['cfnai_lag2'].notna()
auc_lag_random = roc_auc_score(random_test.loc[ok, 'label'], -random_test.loc[ok, 'cfnai_lag2'])
print(f'Random split, test AUC: DistilBERT {roc_auc:.3f} | CFNAI two months before the speech {auc_lag_random:.3f}')

In [ ]:
# The same model, split by time: train 1996-2016, choose the epoch on 2017-2019, test on 2020-2024.
speech_year = merged_df['date'].dt.year
parts = {'train': merged_df[speech_year <= 2016],
         'validation': merged_df[(speech_year > 2016) & (speech_year <= 2019)],
         'test': merged_df[speech_year > 2019]}
for name, part in parts.items():
    print(f'{name:10s} {len(part):5d} speeches, share of class 1: {part["label"].mean():.2f}')

time_data = DatasetDict({name: Dataset.from_pandas(part[['text_cleaned', 'label']], preserve_index=False)
                         for name, part in parts.items()})
time_tokenized = time_data.map(preprocess_function, batched=True)

set_seed(SEED)
time_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels, attn_implementation="eager")
time_args = TrainingArguments(
    output_dir="ModelOutputTime", learning_rate=1e-5, per_device_train_batch_size=32, per_device_eval_batch_size=32,
    num_train_epochs=15, weight_decay=5e-4, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="auc", greater_is_better=True, logging_strategy="epoch",
    seed=SEED, full_determinism=True, report_to="none", save_total_limit=2)
time_trainer = Trainer(model=time_model, args=time_args, train_dataset=time_tokenized["train"],
                       eval_dataset=time_tokenized["validation"], data_collator=data_collator,
                       compute_metrics=compute_metrics)
time_trainer.train()

In [ ]:
time_scores = torch.softmax(torch.tensor(time_trainer.predict(time_tokenized["test"]).predictions), dim=1)[:, 1].numpy()
time_test = parts['test']
auc_text_time = roc_auc_score(time_test['label'], time_scores)
ok = time_test['cfnai_lag2'].notna()
auc_lag_time = roc_auc_score(time_test.loc[ok, 'label'], -time_test.loc[ok, 'cfnai_lag2'])
print(f'Split by time, test AUC: DistilBERT {auc_text_time:.3f} | CFNAI two months before the speech {auc_lag_time:.3f}')

results = pd.DataFrame({'DistilBERT on the speeches': [roc_auc, auc_text_time],
                        'CFNAI two months earlier': [auc_lag_random, auc_lag_time]},
                       index=['Random split', 'Split by time (test 2020-2024)'])
ax = results.plot(kind='bar', rot=0, figsize=(8, 4), color=['tab:purple', 'tab:gray'])
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('Test AUC')
ax.set_ylim(0.3, 0.9)
ax.set_title('The text model against a public number, under two splits')
plt.show()
results.round(3)

The split changes the picture completely. Under the random split, the text model looks useful, although the index of two months earlier already ranks the speeches better. When the test speeches come from later years, the text model does no better than chance (an AUC of 0.5, the dashed line), whereas the public index keeps working. In our runs, DistilBERT reached a test AUC close to 0.44 on 2020 to 2024, and the index close to 0.78. Your numbers will differ somewhat with the GPU.

Every step of the training above is correct, so the lesson concerns usability: the model learned what the data allowed it to learn. The economy of a month is shared by every speech of that month, so under a random split the period of a speech is enough to guess its label, and a speech does not forecast next month's index better than the index itself does.

Three habits follow for any model a bank builds on text. Test it on data from after the training period, as it will be used. Compare it with the numbers the bank already has. Ask whether the target can be read from the input at all. The problem is not specific to this case: [Xue et al. (2026)](https://arxiv.org/abs/2608.17223) found that, across 16 text models for financial news, random splits inflated the scores 1.1 to 6.5 times compared with splits by time. The new Chapter 4 case applies these habits with a target that lives in the text, the monetary policy stance of each speech ([open it in Colab](https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap4_Fed_Stance.ipynb)).

## **Errata**

*Changes from the earlier version of this lab (October 2026).* The results of the earlier version (accuracy, AUC and the January 2025 prediction) are not comparable with the results of this version.

1. **Separate validation and test sets.** The earlier version split the speeches into a training set and a test set only, and also used the test set to evaluate the model during training. The speeches are now split at random into training (70%), validation (15%) and test (15%) sets: the validation set is used during training, and the test set only for the final ROC curve and AUC.

2. **Dates removed from the speech text.** Years and dates written in the speeches (for example, "January 8, 2024") let the model recognize when a speech was given rather than what it says. They are now removed during text cleaning.

3. **Best epoch kept, chosen by validation AUC.** The earlier version used the model from the last (15th) epoch, even after the model had started to overfit. The notebook now keeps the epoch with the highest AUC on the validation set, and reports the ROC curve and AUC on the separate test set, which is used neither for training nor for choosing the epoch.

4. **Seven speeches with wrong dates removed.** Seven speeches in the dataset have a date that differs from the date printed in the speech itself (six are dated one year late). These speeches were matched to the wrong month of CFNAI, and one of them, a January 2024 speech dated December 2024, was the speech used for the January 2025 prediction. These speeches are now dropped, and the prediction uses a genuine December 2024 speech.

5. **Default batch size of 32 instead of 200.** A batch size of 200 runs out of memory on the free Colab GPU (T4). The default is now 32, and the earlier setting is kept as a comment for A100 GPUs. Training with a different batch size gives different results.

6. **Smaller learning rate and weight decay.** The learning rate is now 1e-5 instead of 1e-4, and the weight decay 5e-4 instead of 5e-3.

7. **ROC curve from probabilities, and repeatable runs.** The ROC curve and AUC now use the probability of class 1 (a softmax over the two scores), not the raw score of class 1. The seed is set before the data are split, and `full_determinism=True` makes the training repeatable on the same GPU type.

8. **A usability test added before this list.** A new section retrains the model on the speeches of 1996 to 2016, tests it on 2020 to 2024, and compares it with the index of two months before each speech. The training steps of this notebook are sound, but the test shows that the model is not usable as a forecast. The new Chapter 4 case builds on that lesson.